# Direct questioning — Llama 3.3 70B on αNLI (500 claims, sample hash 24b904e78d)

The model is asked whether the statement is true or false (temperature 0, no facts). Result: 64.6. Run on Kaggle; the API key is read from Kaggle secrets.

In [3]:
# ===== DIRECT QUESTIONING — Llama 3.3 70B + αNLI (same 500-claim sample, no facts) =====
!pip install -q openai datasets scikit-learn

import os, json, time, pickle, hashlib, base64, zipfile, io, urllib.request
from pathlib import Path
import numpy as np, pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)
from IPython.display import HTML, display

EXPECTED_HASH = "24b904e78d"

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
LABEL      = "Llama 3.3 70B"
DATASET    = "aNLI"
RUN_NAME   = "direct_llama_anli"
MAX_TOKENS = 10
client = OpenAI(api_key=UserSecretsClient().get_secret("OPENROUTER_API_KEY"),
                base_url="https://openrouter.ai/api/v1", timeout=180)

WORK_DIR = Path("/kaggle/working") / RUN_NAME
WORK_DIR.mkdir(parents=True, exist_ok=True)
DCK = WORK_DIR / f"{RUN_NAME}_ckpt.pkl"


# ---- SAME αNLI sample as our method (dev split, seed 42) ----
def load_anli_dev():
    try:
        url = "https://storage.googleapis.com/ai2-mosaic/public/abductive-commonsense-reasoning-iclr2020/anli.zip"
        zf = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url, timeout=120).read()))
        names = zf.namelist()
        jl = [n for n in names if n.endswith("dev.jsonl")][0]
        lb = [n for n in names if n.endswith("dev-labels.lst")][0]
        rows = [json.loads(l) for l in zf.read(jl).decode().splitlines() if l.strip()]
        labels = [int(l) for l in zf.read(lb).decode().split()]
        df = pd.DataFrame(rows)
        df["label"] = labels
        print("loaded αNLI dev from the official zip")
        return df[["obs1", "obs2", "hyp1", "hyp2", "label"]]
    except Exception as e:
        print("official zip failed -> Hugging Face fallback:", str(e)[:100])
        from datasets import load_dataset
        ds = load_dataset("allenai/art", split="validation").to_pandas()
        ds = ds.rename(columns={"observation_1": "obs1", "observation_2": "obs2",
                                "hypothesis_1": "hyp1", "hypothesis_2": "hyp2"})
        return ds[["obs1", "obs2", "hyp1", "hyp2", "label"]]


def sentence(s):
    s = str(s).strip()
    return s if s.endswith((".", "!", "?")) else s + "."


raw = load_anli_dev()
raw = raw[raw["label"].isin([1, 2])].copy()
for c in ["obs1", "obs2", "hyp1", "hyp2"]:
    raw[c] = raw[c].astype(str).str.strip()
raw["item_id"] = [hashlib.md5("|".join(r).encode()).hexdigest()[:12]
                  for r in raw[["obs1", "obs2", "hyp1", "hyp2"]].values.tolist()]
raw = raw.drop_duplicates("item_id").sort_values("item_id").reset_index(drop=True)

SEED = 42
items = raw.sample(n=500, random_state=SEED).reset_index(drop=True)
gold_arr = np.array([1] * 250 + [0] * 250)
gold_arr = gold_arr[np.random.RandomState(SEED).permutation(500)]


def make_claim(r, g):
    right = r["hyp1"] if r["label"] == 1 else r["hyp2"]
    wrong = r["hyp2"] if r["label"] == 1 else r["hyp1"]
    hyp = right if g == 1 else wrong
    return f"{sentence(r['obs1'])} {sentence(r['obs2'])} This is because: {sentence(hyp)}"


claims = [make_claim(r, g) for (_, r), g in zip(items.iterrows(), gold_arr)]
gold = gold_arr.astype(int).tolist()
SAMPLE_HASH = hashlib.md5("|".join(f"{s}:{g}" for s, g in
                                   zip(items["item_id"], gold)).encode()).hexdigest()[:10]
print("claims:", len(claims), "| true:", sum(gold), "| SAMPLE HASH:", SAMPLE_HASH)
assert SAMPLE_HASH == EXPECTED_HASH, "Different sample!"


def call(prompt, temperature, max_tokens):
    last = None
    for a in range(1, 9):
        try:
            r = client.chat.completions.create(
                model=MODEL_NAME, temperature=temperature, max_tokens=max_tokens,
                messages=[{"role": "user", "content": prompt}])
            t = r.choices[0].message.content
            if t:
                return t.strip()
            raise ValueError(f"empty (finish_reason={r.choices[0].finish_reason})")
        except Exception as e:
            last = e
            if getattr(e, "status_code", None) == 402 or "Error code: 402" in str(e):
                raise RuntimeError(f"no API credit left: {str(e)[:200]}")
            busy = any(c in str(e) for c in ["429", "502", "503", "504"])
            time.sleep(20 * a if busy else 3 * a)
    raise RuntimeError(f"failed: {last}")


def parse_tf(text):
    t = (text or "").strip().lower().lstrip("*\"' ")
    if t.startswith("true"):
        return 1
    if t.startswith("false"):
        return 0
    has_t, has_f = "true" in t, "false" in t
    if has_t and not has_f:
        return 1
    if has_f and not has_t:
        return 0
    return None


def ask(claim):
    p = ('Is the following statement true or false? Answer with only '
         f'one word: "true" or "false".\n\nStatement: {claim}\n\nAnswer:')
    try:
        return parse_tf(call(p, 0, MAX_TOKENS))
    except RuntimeError as e:
        if "no API credit" in str(e):
            raise
        return None


t = call('Is the following statement true or false? Answer with only one word: '
         '"true" or "false".\n\nStatement: Paris is in France.\n\nAnswer:', 0, MAX_TOKENS)
print("TEST answer:", repr(t), "->", parse_tf(t))
assert parse_tf(t) == 1, "Test failed: tell Claude before running all claims."

dres = pickle.load(open(DCK, "rb")) if DCK.exists() else {}
print("already done:", len(dres))
for i in range(len(claims)):
    if i in dres:
        continue
    dres[i] = ask(claims[i])
    pickle.dump(dres, open(DCK, "wb"))
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(claims)} | unparsed so far: {sum(v is None for v in dres.values())}", flush=True)

pred = [dres[i] if dres[i] is not None else 0 for i in range(len(claims))]
fails = sum(dres[i] is None for i in range(len(claims)))
tn, fp, fn, tp = confusion_matrix(gold, pred, labels=[0, 1]).ravel()
res = pd.DataFrame([{"method": "Direct (no facts)", "n": len(pred),
                     "acc": round(accuracy_score(gold, pred) * 100, 1),
                     "prec": round(precision_score(gold, pred, zero_division=0) * 100, 1),
                     "rec": round(recall_score(gold, pred, zero_division=0) * 100, 1),
                     "f1": round(f1_score(gold, pred, zero_division=0) * 100, 1),
                     "TN": tn, "FP": fp, "FN": fn, "TP": tp}])
pd.set_option("display.width", 220)
print(f"\n=== FINAL: {LABEL} | {DATASET} | Direct | sample {SAMPLE_HASH} ===")
print(res.to_string(index=False))
print(f"unparsed answers (counted as false): {fails}")

csv = WORK_DIR / f"{RUN_NAME}_results.csv"
res.assign(unparsed=fails).to_csv(csv, index=False)
pkl = WORK_DIR / f"{RUN_NAME}_{len(dres)}.pkl"
pickle.dump({"claims": claims, "gold": gold, "pred": dres, "sample_hash": SAMPLE_HASH}, open(pkl, "wb"))
zp = WORK_DIR / f"{RUN_NAME}_backup.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(pkl, arcname=pkl.name)
    z.write(csv, arcname=csv.name)
b64 = base64.b64encode(open(zp, "rb").read()).decode()
display(HTML(
    f'<a download="{zp.name}" href="data:application/zip;base64,{b64}" '
    f'style="display:inline-block;padding:12px 20px;background:#2e7d32;color:white;'
    f'border-radius:6px;text-decoration:none;font-weight:bold;">⬇ Download {zp.name}</a>'))

loaded αNLI dev from the official zip
claims: 500 | true: 250 | SAMPLE HASH: 24b904e78d
TEST answer: 'true' -> 1
already done: 0
  50/500 | unparsed so far: 0
  100/500 | unparsed so far: 0
  150/500 | unparsed so far: 0
  200/500 | unparsed so far: 0
  250/500 | unparsed so far: 0
  300/500 | unparsed so far: 0
  350/500 | unparsed so far: 0
  400/500 | unparsed so far: 0
  450/500 | unparsed so far: 0
  500/500 | unparsed so far: 0

=== FINAL: Llama 3.3 70B | aNLI | Direct | sample 24b904e78d ===
           method   n  acc  prec  rec   f1  TN  FP  FN  TP
Direct (no facts) 500 64.6  80.2 38.8 52.3 226  24 153  97
unparsed answers (counted as false): 0
